# Built a simple bot

## How to integrate LLM into our Graph

In [ ]:
import os
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
#from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Agent's memory
class AgentState(TypedDict): 
    messages: List[HumanMessage]

#the brain
llm =ChatOpenAI(model="gpt-4o")

In [ ]:
def process(state: AgentState) -> AgentState: #LangGraph node
    response =llm.invoke(state["messages"]) #The agent sends the conversation history to the LLM.
    print(f"\nAI:{response.content}") # shows the ai reply
    return state

#### Node:
- receive the current state
- update the state
- return the new state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process") 
graph.add_edge("process", END)

agent = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
user_input= input("Enter: ")
while user_input!= "exit":
    agent.invoke({'messages':[HumanMessage(content=user_input)]})
    user_input= input("Enter: ")

Try giving input and asking the questions. It cannot answer your questions based on your questions

## Let's update

In [ ]:
import os
from typing import TypedDict, List, Dict, Any, Optional, Union
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
class AgentState(TypedDict): 
    messages: List[Union[HumanMessage, AIMessage]]
    
#Human: Hello
#AI: Hi! How can I help?
#Human: Tell me a joke

llm =ChatOpenAI(model="gpt-4o")

def process(state: AgentState) -> AgentState:
    """ Thid nod will solve the request you input"""
    response =llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI:{response.content}")
    print("CURRENT STATE:", state["messages"]) #Shows the full conversation memory.
    return state
#LangGraph passes this updated memory to the next step.

In [ ]:
graph = StateGraph(AgentState) #every step in the graph will use that state.
graph.add_node("process", process) #adding a step to the agent
graph.add_edge(START, "process") 
graph.add_edge("process", END)

agent = graph.compile()

# starting from here it gets different from the previous one

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
conversation_history=[] #to store the chat messages between the user and the AI

user_input= input("Enter: ")
while user_input!= "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result= agent.invoke({'messages':conversation_history})
    conversation_history= result["messages"]
    user_input= input("Enter: ")

### with this version if we exit the bot, it is not going to remember our name again when we run it.

# Let's improve it as a memory based chatbot

In [ ]:
import os
import json
from typing import TypedDict, List, Dict, Any, Optional, Union
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
class AgentState(TypedDict): 
    messages: List[Union[HumanMessage, AIMessage]]

In [ ]:
llm =ChatOpenAI(model="gpt-4o")

def process(state: AgentState) -> AgentState:
    """ Thid nod will solve the request you input"""
    response =llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI:{response.content}")
    #print("CURRENT STATE:", state["messages"])
    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process") 
graph.add_edge("process", END)

agent = graph.compile()

In [ ]:
CONVERSATION_FILE = "conversation.json"

# Load previous conversation if it exists
if os.path.exists(CONVERSATION_FILE):
    with open(CONVERSATION_FILE, "r") as f:
        conversation_history = json.load(f)
        # Convert back to message objects
        conversation_history = [
            HumanMessage(**m) if m["type"] == "human" else AIMessage(**m)
            for m in conversation_history
        ]
else:
    conversation_history = []


In [ ]:
print("Type 'exit' to quit.")
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages": conversation_history})
    conversation_history = result["messages"]  # update state

# Save conversation for next session

with open(CONVERSATION_FILE, "w") as f:
    json.dump([
        {"type": "human", "content": m.content} if isinstance(m, HumanMessage)
        else {"type": "ai", "content": m.content}
        for m in conversation_history
    ], f)

print(f"Conversation saved to {CONVERSATION_FILE}")

## what about creating doc as a node

In [1]:
from typing import TypedDict, List, Union
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]

In [ ]:
#brain
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
CONVERSATION_FILE = "conversation.json"

# Function to load previous conversation (NOT a node) ---
def load_conversation() -> AgentState:
    state: AgentState = {"messages": []}
    if os.path.exists(CONVERSATION_FILE):
        with open(CONVERSATION_FILE, "r") as f:
            data = json.load(f)
        for msg in data:
            if msg["type"] == "human":
                state["messages"].append(HumanMessage(content=msg["content"]))
            elif msg["type"] == "ai":
                state["messages"].append(AIMessage(content=msg["content"]))
    return state

#  Node: process input with LLM ---
def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI: {response.content}")
    return state

#  Node: save conversation ---
def save_conversation(state: AgentState) -> AgentState:
    with open(CONVERSATION_FILE, "w") as f:
        json.dump([
            {"type": "human", "content": m.content} if isinstance(m, HumanMessage)
            else {"type": "ai", "content": m.content}
            for m in state["messages"]
        ], f, indent=2)
    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_node("save", save_conversation)
graph.add_edge(START, "process")
graph.add_edge("process", "save")
graph.add_edge("save", END)

agent = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
state = load_conversation()

print("Type 'exit' to quit.")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    state["messages"].append(HumanMessage(content=user_input))
    state = agent.invoke(state)